In [6]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

# 1. Wyciągamy kolumny ABCD jako macierz numpy
df_train=pd.read_csv("ClassAnnotations.csv")
df_test=pd.read_csv("TestClassAnnotations.csv")
df_val=pd.read_csv("ValClassAnnotations.csv")

train_abcd_values = df_train[['A', 'B', 'C', 'D']].values

# 2. Obliczamy średnią i odchylenie standardowe wzdłuż kolumn (axis=0)
abcd_means = train_abcd_values.mean(axis=0)
abcd_stds = train_abcd_values.std(axis=0)

# Zamieniamy małe wartości odchylenia na coś nieco większego od zera, by uniknąć dzielenia przez zero
abcd_stds = np.where(abcd_stds < 1e-6, 1e-6, abcd_stds)

print(f"Średnie (Train): {abcd_means}")
print(f"Odchylenia (Train): {abcd_stds}")

Średnie (Train): [0.1927001  4.75910318 0.19750723 0.68270492]
Odchylenia (Train): [ 0.10189307 12.65954761  0.0861146   0.05688187]


In [7]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
class_counts = df_train['label'].value_counts().sort_index().values
print(f"Liczebność klas w Train: {class_counts}")

# Obliczamy wagi odwrotnie proporcjonalne do liczebności
class_weights = 1.0 / class_counts

# Mapujemy wagi na każdy konkretny wiersz (obraz) w df_train
sample_weights = [class_weights[label] for label in df_train['label']]
sampler_weights_tensor = torch.DoubleTensor(sample_weights)

# Tworzymy Sampler
train_sampler = WeightedRandomSampler(
    weights=sampler_weights_tensor, 
    num_samples=len(sampler_weights_tensor), 
    replacement=True
)

Liczebność klas w Train: [ 418 1483  173]


In [8]:
class MelanomaMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_dir, abcd_means, abcd_stds, transform=None):
        """
        dataframe: df_train, df_val lub df_test
        img_dir: ścieżka do folderu fizycznymi plikami .jpg
        abcd_means, abcd_stds: statystyki wyliczone na zbiorze TRAIN
        transform: augmentacje wizyjne (torchvision.transforms)
        """
        # Resetujemy indeksy, żeby uniknąć błędów przy .iloc[]
        self.dataframe = dataframe.reset_index(drop=True) 
        self.img_dir = img_dir
        self.transform = transform
        
        self.abcd_means = abcd_means
        self.abcd_stds = abcd_stds

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # 1. Pobranie nazwy i wczytanie obrazu
        img_id = self.dataframe.loc[idx, 'image']
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)

        # 2. Pobranie i STANDARYZACJA parametrów ABCD
        # Pobieramy jako np.float32 z DataFrame
        raw_abcd = self.dataframe.loc[idx, ['A', 'B', 'C', 'D']].values.astype(np.float32)
        
        # Z-score: (x - mean) / std
        norm_abcd = (raw_abcd - self.abcd_means) / self.abcd_stds
        abcd_tensor = torch.tensor(norm_abcd, dtype=torch.float32)

        # 3. Pobranie etykiety (Label)
        label = self.dataframe.loc[idx, 'label']
        label_tensor = torch.tensor(label, dtype=torch.long) # Wymagane dla CrossEntropyLoss

        # Zwracamy tuple: (wejścia_modelu), cel_modelu
        return (image, abcd_tensor), label_tensor

In [10]:
# 1. Definiujemy transformacje (Rozdzielczość pod EfficientNet, np. 224x224)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90), # Bezpieczna, geometryczna augmentacja
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # Średnie ImageNet
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# 2. Inicjalizacja klas Dataset
train_folder="SegDataset/train/trainInput"
val_folder="SegDataset/val/valInput"

train_dataset = MelanomaMultiModalDataset(
    df_train, train_folder, abcd_means, abcd_stds, transform=train_transform)

val_dataset = MelanomaMultiModalDataset(
    df_val, val_folder, abcd_means, abcd_stds, transform=val_test_transform)

# 3. Tworzenie DataLoaderów (ZWRÓĆ UWAGĘ NA SAMPLER W TRENINGU)
batch_size = 32

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    sampler=train_sampler, # <- Twój sampler z poprzedniego kroku!
    drop_last=True         # Przydatne przy Multi-modal, by unikać batchy o wielkości 1
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False # Przy walidacji i testowaniu NIE mieszamy i NIE samplujemy
)

In [11]:
import torch
import torch.nn as nn
import torchvision.models as models

class MelanomaLateFusionModel(nn.Module):
    def __init__(self, num_classes=3, num_tabular_features=4):
        super(MelanomaLateFusionModel, self).__init__()
        
        # ==========================================
        # 1. GAŁĄŹ OBRAZOWA (Image Branch)
        # ==========================================
        # Ładujemy pre-trenowanego EfficientNeta-B0
        weights = models.EfficientNet_B0_Weights.DEFAULT
        self.image_model = models.efficientnet_b0(weights=weights)
        
        # EfficientNet-B0 wypluwa wektor o rozmiarze 1280 przed ostateczną klasyfikacją.
        # Usuwamy oryginalną "głowę" klasyfikującą (zastępujemy ją funkcją tożsamościową),
        # ponieważ chcemy tylko wyciągnąć cechy (features), a nie klasyfikować 1000 klas z ImageNet.
        self.image_model.classifier = nn.Identity()
        
        # ==========================================
        # 2. GAŁĄŹ TABELARYCZNA (Tabular Branch - ABCD)
        # ==========================================
        # Mała sieć, która z 4 parametrów ABCD robi np. 16 cech nieliniowych
        self.tabular_model = nn.Sequential(
            nn.Linear(num_tabular_features, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU()
        )
        
        # ==========================================
        # 3. GŁOWA FUZJI (Fusion & Classification Head)
        # ==========================================
        # Łączymy wektory: 1280 (z obrazu) + 16 (z tabeli) = 1296
        fusion_size = 1280 + 16
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),              # Mocny dropout chroniący przed przeuczeniem
            nn.Linear(fusion_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)     # Ostateczne wyjście na 3 klasy (MEL, NV, BKL)
        )

    def forward(self, image, tabular):
        # 1. Przepuszczamy obraz przez EfficientNet
        img_features = self.image_model(image) # Kształt: [Batch, 1280]
        
        # 2. Przepuszczamy ABCD przez małą sieć
        tab_features = self.tabular_model(tabular) # Kształt: [Batch, 16]
        
        # 3. Fuzja (Konkatenacja obu wektorów wzdłuż wymiaru cech)
        fused_features = torch.cat((img_features, tab_features), dim=1) # Kształt: [Batch, 1296]
        
        # 4. Ostateczna klasyfikacja
        output = self.classifier(fused_features) # Kształt: [Batch, 3]
        
        return output

# ==========================================
# TEST MODELU (Sanity Check)
# ==========================================
if __name__ == "__main__":
    # Inicjalizacja modelu
    model = MelanomaLateFusionModel(num_classes=3)
    
    # Tworzymy sztuczny batch danych (np. 8 obrazów) żeby sprawdzić czy wszystko działa
    dummy_images = torch.randn(8, 3, 224, 224) # 8 obrazków RGB 224x224
    dummy_abcd = torch.randn(8, 4)             # 8 wektorów ABCD
    
    # Przepuszczamy przez model
    predictions = model(dummy_images, dummy_abcd)
    
    print(f"Kształt wejścia obrazów: {dummy_images.shape}")
    print(f"Kształt wejścia ABCD: {dummy_abcd.shape}")
    print(f"Kształt wyjścia modelu: {predictions.shape}") # Powinno być [8, 3]
    print("Jeśli tu dotarłeś bez błędu, Twoja architektura działa perfekcyjnie!")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /home/amay/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 23.1MB/s]


Kształt wejścia obrazów: torch.Size([8, 3, 224, 224])
Kształt wejścia ABCD: torch.Size([8, 4])
Kształt wyjścia modelu: torch.Size([8, 3])
Jeśli tu dotarłeś bez błędu, Twoja architektura działa perfekcyjnie!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm # Do ładnego paska postępu

# ==========================================
# 1. PRZYGOTOWANIE ŚRODOWISKA
# ==========================================
# Automatyczne wykrywanie karty graficznej (NVIDIA / Apple Silicon / CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Rozpoczynam trening na urządzeniu: {device}")

# Inicjalizacja modelu z poprzedniego kroku (3 klasy: MEL, NV, BKL)
model = MelanomaLateFusionModel(num_classes=3).to(device)

# Funkcja straty dla klasyfikacji wieloklasowej
criterion = nn.CrossEntropyLoss()

# Optymalizator (AdamW świetnie sprawdza się w sieciach wizyjnych)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

# Scheduler (Zmniejsza Learning Rate, gdy model przestaje się uczyć)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, verbose=True)

# ==========================================
# 2. USTAWIENIA TRENINGU
# ==========================================
num_epochs = 20
best_val_loss = float('inf')
patience = 5 # Early Stopping
epochs_no_improve = 0

print("Rozpoczynamy trening!")
print("-" * 30)

# ==========================================
# 3. GŁÓWNA PĘTLA
# ==========================================
for epoch in range(num_epochs):
    
    # ------------------ FAZA TRENINGU ------------------
    model.train() # Włącza Dropout i BatchNorm
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    # Pasek postępu dla epoki
    loop = tqdm(train_loader, leave=False)
    for (images, tabular), labels in loop:
        # Przeniesienie danych na kartę graficzną
        images = images.to(device)
        tabular = tabular.to(device)
        labels = labels.to(device)
        
        # Zerowanie gradientów
        optimizer.zero_grad()
        
        # Forward pass (Przepuszczenie danych przez model)
        outputs = model(images, tabular)
        
        # Obliczenie błędu
        loss = criterion(outputs, labels)
        
        # Backward pass (Obliczenie gradientów)
        loss.backward()
        
        # Aktualizacja wag
        optimizer.step()
        
        # Statystyki
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
        # Aktualizacja opisu w pasku postępu
        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())
        
    avg_train_loss = train_loss / len(train_loader.dataset)
    train_acc = correct_train / total_train

    # ------------------ FAZA WALIDACJI ------------------
    model.eval() # Wyłącza Dropout, zamraża BatchNorm
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad(): # Wyłączamy liczenie gradientów (oszczędza pamięć RAM)
        for (images, tabular), labels in val_loader:
            images, tabular, labels = images.to(device), tabular.to(device), labels.to(device)
            
            outputs = model(images, tabular)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    avg_val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct_val / total_val
    
    # Krok schedulera (na podstawie błędu walidacyjnego)
    scheduler.step(avg_val_loss)
    
    # Wypisywanie statystyk po każdej epoce
    print(f"Epoka {epoch+1}/{num_epochs} | "
          f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    # ------------------ ZAPISYWANIE MODELU & EARLY STOPPING ------------------
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        # Zapisujemy wagi modelu do pliku
        torch.save(model.state_dict(), "best_melanoma_model.pth")
        print("  -> Nowy najlepszy model zapisany!")
    else:
        epochs_no_improve += 1
        print(f"  -> Brak poprawy od {epochs_no_improve} epok.")
        
        if epochs_no_improve >= patience:
            print("Early Stopping! Przerywam trening, aby uniknąć przeuczenia.")
            break

print("-" * 30)
print("Trening zakończony! Najlepsze wagi znajdują się w pliku 'best_melanoma_model.pth'.")